In [ ]:
# Setup the Jupyter version of Dash
from dash import Dash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64

# Data handling
import pandas as pd

from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

db = AnimalShelter()
db.create_indexes()

outcome_pipeline = [
    {'$group': {'_id': '$outcome_type', 'count': {'$sum': 1}}},
    {'$sort': {'count': -1}}
]

outcome_summary = pd.DataFrame(db.aggregate(outcome_pipeline))
outcome_summary = outcome_summary.rename(columns={'_id': 'outcome_type'})

outcome_fig = px.bar(
    outcome_summary,
    x='outcome_type',
    y='count',
    title='Outcome Type Summary'
)

outcome_fig.update_layout(
    margin=dict(l=10, r=10, t=40, b=10),
    height=300,
    yaxis_title='Outcome Type',
    xaxis_title=''
)

df = pd.DataFrame.from_records(db.read())

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
with open(image_filename, 'rb') as image_file:
    encoded_image = base64.b64encode(image_file.read())

app.layout = html.Div([
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '120px', 'marginRight': '12px'}
            ),
            href='https://www.snhu.edu', target='_blank', rel='noopener noreferrer'
        ),
        html.Div([
            html.B('Grazioso Salvare Dashboard'),
            html.Div('Built by Caleb Luplow - Enhanced Artifact', style={'opacity': 0.75})
        ], style={'display': 'inline-block', 'verticalAlign': 'middle', 'marginLeft': '8px'})
    ], style={'display': 'flex', 'alignItems': 'center'}),
    
    html.Hr(),
    
    html.Div(
        style={
            'display': 'flex',
            'gap': '20px',
            'alignItems': 'flex-start',
            'marginBottom': '20px'
        },
        children=[
            html.Div(
                style={
                    'flex': '2',
                    'minWidth': '0',
                    'padding': '10px',
                    'border': '1px solid #ddd',
                    'borderRadius': '8px',
                    'backgroundColor': '#fafafa'
                },
                children=[
                    dcc.Graph(
                        figure=outcome_fig,
                        config={'displayModeBar': False}
                    )
                ]
            ),
            html.Div(
                style={
                    'flex': '1',
                    'padding': '16px',
                    'border': '1px solid #ddd',
                    'borderRadius': '8px',
                    'backgroundColor': '#fafafa'
                },
                children=[
                    html.H4('Rescue Type', style={'marginTop': '0'}),
                    dcc.RadioItems(
                        id='filter-type',
                        options=[
                            {'label': 'Water Rescue', 'value': 'water'},
                            {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                            {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                            {'label': 'Reset (All)', 'value': 'reset'}
                        ],
                        value='reset',
                        labelStyle={'display': 'inline-block', 'marginRight': '18px'}
                    )
                ]
            )
        ]
    ),
        
    html.Hr(),
    
    dash_table.DataTable(
        id='datatable-id',
        columns=[{'name': i, 'id': i, 'deletable': False, 'selectable': True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'minWidth': '120px', 'width': '120px', 'maxWidth': '240px',
            'whiteSpace': 'normal'
        },
        style_header={'fontWeight': 'bold'}
    ),
    
    html.Br(),
    html.Hr(),
    
    html.Div(className='row',
        style={'display' : 'flex'}, children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
    ])
])


#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if not filter_type or filter_type == 'reset':
        query = {}
        
    elif filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                {'breed': {'$regex': r'Labrador Retriever', '$options': 'i'}},
                {'breed': 'Chesapeake Bay Retriever'},
                {'breed': 'Newfoundland'}
            ]}
        
    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            'breed': {'$in': [
                'German Shepherd',
                'Alaskan Malamute',
                'Old English Sheepdog',
                'Siberian Husky',
                'Rottweiler'
            ]}
        }
        
    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            'breed': {'$in': [
                'Doberman Pinscher',
                'German Shepherd',
                'Golden Retriever',
                'Bloodhound',
                'Rottweiler'
            ]}
        }
        
    else:
        query = {}
        
    records = db.read(query)
    
    dff = pd.DataFrame.from_records(records)
    
    if dff.empty:
        return []
    
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
        
    return dff.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.Div('No data to display.')]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    breed_col = 'breed' if 'breed' in dff.columns else dff.columns[0]
    counts = dff[breed_col].value_counts().reset_index()
    counts.columns = [breed_col, 'count']
    counts = counts.head(15)
    
    fig = px.bar(
        counts,
        x='count',
        y=breed_col,
        orientation='h',
        title='Breed Distribution (Top 15 in Current View)'
    )
    fig.update_layout(margin=dict(l=10, r=10, t=40, b=10), height=460)
    
    return [dcc.Graph(figure=fig)]
    

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return [html.Div('No data available')]
    
    dff = pd.DataFrame.from_dict(viewData)
    row = 0 if not index else index[0]
        
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']
    
    if pd.isna(lat) or pd.isna(lon):
        return [html.Div('Location data is not available for this animal.')]
        
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H1(str(name) if pd.notna(name) else 'Unnamed Animal'),
                            html.P(str(breed))
                ])
            ])
        ])
    ]



app.run(debug=True)
print('Open http://localhost:8050')


Open http://localhost:8050
